In [9]:
from google.colab import drive
drive.mount('/content/drive')

# Notice the quotes around the Drive path to handle the spaces!
!cp "/content/drive/MyDrive/DLCV Project Code/DLCV_Project_Code.zip" /content/
!unzip -q -o /content/DLCV_Project_Code.zip -d /content/workspace
%cd /content/workspace

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content/workspace


In [10]:
import json
import os

# ---> PASTE YOUR KAGGLE INFO HERE <---
kaggle_creds = {
    "username": "srinathkaggle",
    "key": "KGAT_34c34e54f710de7fa51d6b514f81b9c8"
}

# 1. Setup the Kaggle config folder in the root directory
os.makedirs('/root/.kaggle', exist_ok=True)

# 2. Write the credentials to the json file
with open('/root/.kaggle/kaggle.json', 'w') as f:
    json.dump(kaggle_creds, f)

# 3. Set the required permissions (Kaggle CLI will fail without this)
os.chmod('/root/.kaggle/kaggle.json', 0o600)

# 4. Download and unzip using the exact name from your URL
!mkdir -p data
!kaggle datasets download -d masoudnickparvar/brain-tumor-mri-dataset -p data
!cd data && unzip -q -o brain-tumor-mri-dataset.zip && cd ..

# 5. Install dependencies
!pip install -r requirements.txt

Dataset URL: https://www.kaggle.com/datasets/masoudnickparvar/brain-tumor-mri-dataset
License(s): Attribution 4.0 International (CC BY 4.0)
 99% 156M/157M [00:00<00:00, 1.63GB/s]
100% 157M/157M [00:00<00:00, 1.63GB/s]


In [11]:
!echo "=== STARTING EFFICIENTNET BASELINE: $(date) ==="
!python train.py --model efficientnet_b0 --epochs 30 --num_workers 4 --batch_size 64

!echo "=== STARTING EFFICIENTNET SAM: $(date) ==="
!python train.py --model efficientnet_b0 --use_sam --rho 0.05 --epochs 30 --num_workers 4 --batch_size 64

!echo "=== STARTING RESNET-50: $(date) ==="
!python train.py --model resnet50 --epochs 30 --num_workers 4 --batch_size 64

!echo "=== ALL TRAINING FINISHED: $(date) ==="

=== STARTING EFFICIENTNET BASELINE: Wed Mar 11 03:16:11 PM UTC 2026 ===
Using device: cuda
Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b0_rwightman-7f5810bc.pth
100% 20.5M/20.5M [00:00<00:00, 156MB/s]
Epoch 1/30  train_loss=0.3671  val_macro_f1=0.8470  val_acc=0.8454
  -> saved /content/workspace/outputs/checkpoints/best_efficientnet_b0_baseline.pt
Epoch 2/30  train_loss=0.1257  val_macro_f1=0.9106  val_acc=0.9102
  -> saved /content/workspace/outputs/checkpoints/best_efficientnet_b0_baseline.pt
Epoch 3/30  train_loss=0.0707  val_macro_f1=0.9528  val_acc=0.9528
  -> saved /content/workspace/outputs/checkpoints/best_efficientnet_b0_baseline.pt
Epoch 4/30  train_loss=0.0501  val_macro_f1=0.9509  val_acc=0.9509
Epoch 5/30  train_loss=0.0328  val_macro_f1=0.9676  val_acc=0.9676
  -> saved /content/workspace/outputs/checkpoints/best_efficientnet_b0_baseline.pt
Epoch 6/30  train_loss=0.0243  val_

In [12]:
# Saving back to the new folder with quotes
!cp -r /content/workspace/outputs/checkpoints "/content/drive/MyDrive/DLCV Project Code/"

In [13]:
# 1. Evaluate EfficientNet Baseline
!python eval.py --checkpoint outputs/checkpoints/best_efficientnet_b0_baseline.pt --output_dir outputs/results/baseline

# 2. Evaluate EfficientNet SAM
!python eval.py --checkpoint outputs/checkpoints/best_efficientnet_b0_sam.pt --output_dir outputs/results/sam

# 3. Evaluate ResNet-50
!python eval.py --checkpoint outputs/checkpoints/best_resnet50_baseline.pt --output_dir outputs/results/resnet50

Using device: cuda
Saved outputs/results/baseline/metrics_best_efficientnet_b0_baseline.json
Test results:
  Accuracy:       0.9870
  Macro-F1:       0.9871
  Macro-Precision:0.9874
  Macro-Recall:   0.9870
  AUROC:          0.9995
Confusion matrix:
[[261   9   0   0]
 [  1 268   0   1]
 [  0   0 270   0]
 [  0   3   0 267]]
Saved outputs/results/baseline/metrics_best_efficientnet_b0_baseline.md
Using device: cuda
Saved outputs/results/sam/metrics_best_efficientnet_b0_sam.json
Test results:
  Accuracy:       0.9889
  Macro-F1:       0.9889
  Macro-Precision:0.9890
  Macro-Recall:   0.9889
  AUROC:          0.9992
Confusion matrix:
[[266   4   0   0]
 [  2 267   0   1]
 [  1   0 269   0]
 [  1   3   0 266]]
Saved outputs/results/sam/metrics_best_efficientnet_b0_sam.md
Using device: cuda
Saved outputs/results/resnet50/metrics_best_resnet50_baseline.json
Test results:
  Accuracy:       0.9824
  Macro-F1:       0.9824
  Macro-Precision:0.9826
  Macro-Recall:   0.9824
  AUROC:          0.99

In [16]:
!ls -R /content/workspace

/content/workspace:
config.py   explain.py	models.py	  sam.py	 verify_data.py
data	    losses.py	outputs		  train.py
dataset.py  __MACOSX	__pycache__	  transforms.py
eval.py     metrics.py	requirements.txt  utils.py

/content/workspace/data:
brain-tumor-mri-dataset.zip  Testing  Training

/content/workspace/data/Testing:
glioma	meningioma  notumor  pituitary

/content/workspace/data/Testing/glioma:
Te-gl_100.jpg  Te-gl_173.jpg  Te-gl_245.jpg  Te-gl_317.jpg  Te-gl_38.jpg
Te-gl_101.jpg  Te-gl_174.jpg  Te-gl_246.jpg  Te-gl_318.jpg  Te-gl_390.jpg
Te-gl_102.jpg  Te-gl_175.jpg  Te-gl_247.jpg  Te-gl_319.jpg  Te-gl_391.jpg
Te-gl_103.jpg  Te-gl_176.jpg  Te-gl_248.jpg  Te-gl_31.jpg   Te-gl_392.jpg
Te-gl_104.jpg  Te-gl_177.jpg  Te-gl_249.jpg  Te-gl_320.jpg  Te-gl_393.jpg
Te-gl_105.jpg  Te-gl_178.jpg  Te-gl_24.jpg   Te-gl_321.jpg  Te-gl_394.jpg
Te-gl_106.jpg  Te-gl_179.jpg  Te-gl_250.jpg  Te-gl_322.jpg  Te-gl_395.jpg
Te-gl_107.jpg  Te-gl_17.jpg   Te-gl_251.jpg  Te-gl_323.jpg  Te-gl_396.jpg
Te-gl_1

In [15]:
import os
import sys

# Add the current directory to the path so it can find the explainability module
os.environ['PYTHONPATH'] = "/content/workspace:" + os.environ.get('PYTHONPATH', '')

# Now run the explainability script
!python explain.py --checkpoint outputs/checkpoints/best_efficientnet_b0_sam.pt --output_dir outputs/results/explainability

Traceback (most recent call last):
  File "/content/workspace/explain.py", line 19, in <module>
    from explainability import GradCAM, IntegratedGradients
ModuleNotFoundError: No module named 'explainability'


In [17]:
# Sync both checkpoints and result visualizations
!cp -r /content/workspace/outputs/checkpoints "/content/drive/MyDrive/DLCV Project Code/"
!cp -r /content/workspace/outputs/results "/content/drive/MyDrive/DLCV Project Code/"
!echo "=== ALL FILES SECURED TO DRIVE: $(date) ==="

=== ALL FILES SECURED TO DRIVE: Wed Mar 11 03:37:27 PM UTC 2026 ===
